# Passo 1 – Reunir e organizar as bases

Este notebook:

1. Baixa as três bases dos repositórios oficiais no GitHub.
2. Lê cada base e coloca todas no mesmo formato: uma coluna de texto e uma de rótulo.
3. Mostra quantos textos há por base e por classe.
4. Faz um diagnóstico: mede o tamanho dos textos e a presença de palavras típicas de agências de checagem em cada classe.
5. Salva a tabela unificada em `dados/processados/bases_unificadas.csv`.

**Rótulo usado em todo o trabalho:** `1` = falso, `0` = verdadeiro.

## 1. Preparação (mesma do Passo 0)

In [ ]:
import os, subprocess, glob
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')

PASTAS = {
    'brutos': os.path.join(PASTA_TCC, 'dados', 'brutos'),
    'processados': os.path.join(PASTA_TCC, 'dados', 'processados'),
    'resultados': os.path.join(PASTA_TCC, 'resultados'),
}
for caminho in PASTAS.values():
    os.makedirs(caminho, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option('display.max_colwidth', 120)

## 2. Baixar as bases

| Base | Repositório oficial | Licença |
|---|---|---|
| Fake.Br | github.com/roneysco/Fake.br-Corpus | ver README do repositório |
| FakeRecogna | github.com/Gabriel-Lino-Garcia/FakeRecogna | ver README do repositório |
| FactPolCheckBr | github.com/Interfaces-UFSCAR/Dataset-FactPolCheckBr | CC BY-NC-SA 4.0 |

Se a pasta já existir no Drive, o download é pulado. A versão baixada (código do commit) é registrada, para identificar exatamente qual versão foi usada.

In [ ]:
REPOS = {
    'Fake.br-Corpus': 'https://github.com/roneysco/Fake.br-Corpus',
    'FakeRecogna': 'https://github.com/Gabriel-Lino-Garcia/FakeRecogna',
    'Dataset-FactPolCheckBr': 'https://github.com/Interfaces-UFSCAR/Dataset-FactPolCheckBr',
}

versoes_bases = {}
for nome, url in REPOS.items():
    destino = os.path.join(PASTAS['brutos'], nome)
    if not os.path.isdir(destino):
        print(f'Baixando {nome}...')
        subprocess.run(['git', 'clone', '--depth', '1', '-q', url, destino], check=True)
    commit = subprocess.run(['git', '-C', destino, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
    data = subprocess.run(['git', '-C', destino, 'log', '-1', '--format=%cs'], capture_output=True, text=True).stdout.strip()
    versoes_bases[nome] = f'commit {commit[:7]} de {data}'
    print(f'{nome:24s} {versoes_bases[nome]}')

with open(os.path.join(PASTAS['resultados'], 'versoes_bases.txt'), 'w', encoding='utf-8') as f:
    for nome, v in versoes_bases.items():
        f.write(f'{nome}: {REPOS[nome]} ({v})\n')

## 3. Fake.Br

Cada notícia está num arquivo `.txt`. A notícia falsa e a verdadeira correspondente têm o mesmo número, o que forma o par.

O repositório traz duas versões dos textos:
- `full_texts`: textos completos;
- `size_normalized_texts`: em cada par, o texto maior foi cortado para ficar do tamanho do menor.

A célula abaixo compara as duas, para justificar qual será usada.

In [ ]:
raiz_fakebr = os.path.join(PASTAS['brutos'], 'Fake.br-Corpus')

def ler_fakebr(versao):
    linhas = []
    for classe, rotulo in [('fake', 1), ('true', 0)]:
        for arq in glob.glob(os.path.join(raiz_fakebr, versao, classe, '*.txt')):
            numero = int(os.path.basename(arq).split('.')[0])
            with open(arq, encoding='utf-8', errors='replace') as f:
                linhas.append({'texto': f.read().strip(), 'rotulo': rotulo, 'par_id': f'fakebr_{numero}'})
    return pd.DataFrame(linhas)

fakebr_completo = ler_fakebr('full_texts')
fakebr = ler_fakebr('size_normalized_texts')

def palavras(serie):
    return serie.str.split().str.len()

comparacao = pd.DataFrame({
    'versão completa': fakebr_completo.groupby('rotulo')['texto'].apply(lambda s: palavras(s).median()),
    'versão normalizada': fakebr.groupby('rotulo')['texto'].apply(lambda s: palavras(s).median()),
}).rename(index={1: 'falsa', 0: 'verdadeira'})
print('Mediana de palavras por notícia no Fake.Br:')
comparacao

## 4. FakeRecogna

A base vem numa planilha. Atenção: na base original, `Classe` = 0 é falso e 1 é verdadeiro. Aqui o rótulo é invertido para seguir o padrão do trabalho (1 = falso).

O repositório tem dois arquivos. Os dois trazem o texto já pré-processado pelos autores (palavras reduzidas à forma básica e, no arquivo principal, sem stopwords). O texto original das notícias não está disponível no repositório.

In [ ]:
raiz_recogna = os.path.join(PASTAS['brutos'], 'FakeRecogna', 'dataset')
recogna_bruto = pd.read_excel(os.path.join(raiz_recogna, 'FakeRecogna.xlsx'))
print('Linhas lidas:', len(recogna_bruto))

recogna = pd.DataFrame({
    'texto': recogna_bruto['Noticia'].astype('string').str.strip(),
    'rotulo': recogna_bruto['Classe'].map({0: 1, 1: 0}),
    'titulo': recogna_bruto['Titulo'].astype('string').str.strip(),
    'categoria': recogna_bruto['Categoria'],
    'fonte': recogna_bruto['URL'],
})
print('Linhas sem rótulo:', recogna['rotulo'].isna().sum())
recogna.sample(3, random_state=SEMENTE)[['titulo', 'texto', 'rotulo']]

## 5. FactPolCheckBr

O arquivo `com_texto.csv` reúne as checagens das dez agências e traz o texto de cada checagem.

Atenção: o campo `texto` é a **matéria da agência de checagem**, não o boato original. A coluna `Natureza da notícia` traz a classificação dada pela agência.

In [ ]:
raiz_fpc = os.path.join(PASTAS['brutos'], 'Dataset-FactPolCheckBr', 'dados')
fpc_bruto = pd.read_csv(os.path.join(raiz_fpc, 'com_texto.csv'))
print('Linhas lidas:', len(fpc_bruto))
print('\nClassificação dada pelas agências:')
print(fpc_bruto['Natureza da notícia'].value_counts(dropna=False).to_string())

fpc = pd.DataFrame({
    'texto': fpc_bruto['texto'].astype('string').str.strip(),
    'rotulo': fpc_bruto['Natureza da notícia'].map({'Falsa': 1, 'Verdadeira': 0}),
    'titulo': fpc_bruto['Título da checagem'].astype('string').str.strip(),
    'categoria': fpc_bruto['Candidato(s) favorecidos(s) pela notícia falsa'],
    'fonte': fpc_bruto['Link'],
})
fpc.sample(3, random_state=SEMENTE)[['titulo', 'texto', 'rotulo']]

## 6. Juntar as três bases numa tabela única

Colunas finais: `base`, `texto`, `rotulo`, `par_id` (só Fake.Br), `titulo`, `categoria` e `fonte`.

Nesta etapa nada é descartado. A limpeza (textos vazios, curtos e duplicados) é o Passo 2.

In [ ]:
fakebr['base'] = 'Fake.Br'
recogna['base'] = 'FakeRecogna'
fpc['base'] = 'FactPolCheckBr'

colunas = ['base', 'texto', 'rotulo', 'par_id', 'titulo', 'categoria', 'fonte']
bases = pd.concat([fakebr, recogna, fpc], ignore_index=True).reindex(columns=colunas)

contagem = (bases.assign(classe=bases['rotulo'].map({1: 'falso', 0: 'verdadeiro'}).fillna('sem rótulo'))
                 .pivot_table(index='base', columns='classe', values='texto', aggfunc='size', fill_value=0))
contagem['total'] = contagem.sum(axis=1)
contagem = contagem.loc[['Fake.Br', 'FakeRecogna', 'FactPolCheckBr']]
contagem.to_csv(os.path.join(PASTAS['resultados'], 'contagem_bases.csv'))
print('Textos por base e por classe (antes da limpeza):')
contagem

## 7. Diagnóstico: palavras típicas de agências de checagem

Pergunta: o texto das notícias falsas traz palavras que entregam o rótulo, como "boato", "falso" ou "checagem"?

Se essas palavras aparecem muito mais na classe falsa, o modelo pode aprender a reconhecer **o estilo da agência de checagem** em vez dos padrões da desinformação. A tabela mostra a porcentagem de textos de cada classe que contém cada palavra.

In [ ]:
MARCAS = ['boato', 'falso', 'fake', 'checagem', 'checar', 'verific', 'circula', 'enganos']

def diagnostico(df):
    texto = df['texto'].fillna('').str.lower()
    linhas = {}
    for marca in MARCAS:
        linhas[marca] = {
            classe: round(texto[df['rotulo'] == r].str.contains(marca, regex=False).mean() * 100, 1)
            for classe, r in [('falso', 1), ('verdadeiro', 0)]
        }
    return pd.DataFrame(linhas).T

partes = []
for nome in ['Fake.Br', 'FakeRecogna', 'FactPolCheckBr']:
    d = diagnostico(bases[bases['base'] == nome])
    d.columns = pd.MultiIndex.from_product([[nome], d.columns])
    partes.append(d)
diag = pd.concat(partes, axis=1)
diag.to_csv(os.path.join(PASTAS['resultados'], 'diagnostico_marcas_checagem.csv'))
print('% de textos que contêm cada palavra (antes do pré-processamento):')
diag

## 8. Salvar a tabela unificada

In [ ]:
saida = os.path.join(PASTAS['processados'], 'bases_unificadas.csv')
bases.to_csv(saida, index=False)
print(f'{len(bases)} linhas salvas em {saida}')

## O que levar para o texto (Seção 4.3)

- A versão (commit) de cada base, salva em `resultados/versoes_bases.txt`.
- A tabela de contagem, salva em `resultados/contagem_bases.csv`. Ela substitui o Quadro 1.
- A comparação de tamanho do Fake.Br, que justifica o uso da versão normalizada.
- O diagnóstico de palavras de checagem, que fundamenta as decisões sobre FakeRecogna e FactPolCheckBr.